# ⚙️ LIGO Engineering Notes

---

## 📑 Table of Contents

- [Loggers](#loggers)
- [GPS Time](#gps-time)
- [MiB vs. MB](#mib-vs-mb)
- [Spark](#spark)

---

## Loggers

Python's `logging` library provides **levels** that control how much gets logged. Each level has a string name, a numeric value, and a constant.

| Level | Numeric Value | Recommended For |
|-------|--------------|-----------------|
| DEBUG | 10 | Development — logs everything |
| INFO | 20 | Production — logs normal operations |
| WARNING | 30 | Potential issues |
| ERROR | 40 | Errors that need attention |
| CRITICAL | 50 | Catastrophic events only |

You can define the level in three equivalent ways:

```python
level="DEBUG"        # string
level=10             # numeric value
level=logging.DEBUG  # constant
```

> **CRITICAL** has the highest numeric value — it is the strictest filter, silencing almost all output and only letting catastrophic events through.

### Why `stream=sys.stdout`?

By default, Python sends all logs to **Standard Error** (`sys.stderr`). Setting `stream=sys.stdout` redirects logs to **Standard Output** instead.

This matters because external systems (like log collectors, CI pipelines, or containers) often treat `stderr` as an error signal. Using `sys.stdout` forces logs down the "normal data" pipe so external systems don't misinterpret them as failures.

```python
logging.basicConfig(level=logging.DEBUG, stream=sys.stdout)
```

---

## GPS Time

> **GPS time** is a continuous integer count of seconds since **January 6, 1980 00:00:00 UTC**.

- It has **no leap seconds** — it never pauses or resets
- It increases monotonically, making it ideal for scientific timestamps
- LIGO uses GPS time to precisely mark every sample in the strain data

---

## MiB vs. MB

Two different standards — both correct, but for different contexts:

| Unit | Base | 1 unit = | Used in |
|------|------|----------|---------|
| MB (Megabyte) | Powers of 10 | 1,000,000 bytes | Marketing, storage, internet speeds |
| MiB (Mebibyte) | Powers of 2 | 2²⁰ = 1,048,576 bytes | OS, memory, low-level computing |

**Why LIGO uses MiB:** computers are binary, and memory naturally aligns with powers of 2 — 2¹⁰ = 1024, 2²⁰, 2³⁰, etc.

### LIGO Example — 4096 sec file at 4096 Hz

- Each sample is **float64** = 64 bits = **8 bytes**
- Total samples: `4096 sec × 4096 Hz = 16,777,216 samples`

```
16,777,216 × 8 bytes = 134,217,728 bytes

÷ 1,000,000     = 134 MB
÷ 1,048,576     = 128 MiB
```

Both are correct — they just use different divisors.

---

## Spark

### Data Size in LIGO

For a **4096 sec file at 4096 Hz**, as shown above, the raw strain data is **134 MB / 128 MiB**.

When storing this in Spark, there are two design choices:

| Option | Description | Recommended? |
|--------|-------------|:---:|
| One giant row | Store entire strain as 1 row with metadata | ❌ No |
| Windowed rows | Split into 2-second windows, one row per window | ✅ Yes |

---

### Why Is One 134 MB Row Bad in Spark?

Spark's parallelism comes from **partitions**:

```
1 partition  →  1 task  →  1 CPU core
```

Executors run **many tasks in parallel** across many cores. So parallelism depends entirely on having **many partitions**.

If the entire signal is stored as **one giant row**:
- The DataFrame has essentially one logical object
- A UDF sees **one row → one function call → one task → one core**
- The rest of your cluster sits idle

> **Spark likes many small/medium records. It dislikes one giant record.**

Even if your cluster has 100 cores, only 1 core does the real work on that signal.

**The fix — windowing:**

Split the signal into 2-second windows before storing. Each window becomes its own row, and Spark can process all windows in parallel across the cluster.

```
4096 sec ÷ 2 sec/window = 2048 rows  →  2048 tasks  →  full parallelism
```

---

### Ideal Partition Size

| Data Type | Recommended Size per Partition |
|-----------|-------------------------------|
| General objects | 100 MB to 256 MB |
| Parquet files | 128 MB to 1 GB |

**LIGO — 2-second window at 4096 Hz:**

```
8192 samples × 8 bytes (float64) = 65,536 bytes ≈ 64 KiB per row
```

64 KiB per row is well within the reasonable range — small enough for Spark to handle many in parallel, large enough to avoid excessive overhead per task.

---

### Spark Task

> A **task** is the smallest unit of work in Spark: one operation applied to one partition.

```
Task = Partition + Operation
```

One task runs on one CPU core. The more partitions you have, the more tasks Spark can distribute across your cluster — and the faster your job runs.